# StreamSense — Serving the Registered Models (no retraining)

**Production pattern:** the models were trained **once** in the training half and committed to
`models/` (a POC stand-in for a model registry). This notebook **loads those exact model files
and serves them** — it does **not** retrain, and it does **not** re-evaluate. The single A/B
result (**+6.3% NDCG@10**, ranker vs popularity) belongs to the training half; the model served
here is byte-identical to the one that produced it.

Serves: **TensorFlow Serving** (retrieval), **TorchServe** (ranker), **Triton/ONNX** (ranker).
Runs on Colab's Python 3.12 — no TFX/TFRS here (those belong to the training pipeline, run
separately on a Python ≤3.11 environment).

## 0. Environment

In [ ]:
import sys
print("Python:", sys.version.split()[0], "(serving needs no TFX, so 3.12 is fine)")
!nvidia-smi -L || echo "CPU only is fine for serving this small model" 

## 1. Clone the repo (now ships the registered models in `models/`)

In [ ]:
!git clone https://github.com/techykajal/streamsense-ott-recsys.git
%cd streamsense-ott-recsys
print("Registered models:")
!ls -R models | head -30

## 2. Install serving dependencies only\nNo TFX, no TensorFlow-Recommenders, no retraining — just what's needed to *serve*.

In [ ]:
!pip install -q torch torchserve torch-model-archiver onnxruntime
print("Serving deps installed.")

## 3. TensorFlow Serving — serve the **trained** retrieval model
Loads the committed SavedModel directly (TF Serving is a binary; it needs no Python TF).

In [ ]:
!echo "deb http://storage.googleapis.com/tensorflow-serving-apt stable tensorflow-model-server tensorflow-model-server-universal" \
   | tee /etc/apt/sources.list.d/tensorflow-serving.list >/dev/null
!curl -s https://storage.googleapis.com/tensorflow-serving-apt/tensorflow-serving.release.pub.gpg | apt-key add - 2>/dev/null
!apt-get update -qq && apt-get install -y -qq tensorflow-model-server

# Stage the REGISTERED SavedModel as <name>/<version>/ (no training — just a copy)
!rm -rf /content/serve && mkdir -p /content/serve/retrieval/1 && cp -r models/retrieval/* /content/serve/retrieval/1/
import subprocess, time
subprocess.Popen(["tensorflow_model_server","--rest_api_port=8501","--model_name=retrieval",
                  "--model_base_path=/content/serve/retrieval"],
                 stdout=open("/content/tfserve.log","w"), stderr=subprocess.STDOUT)
time.sleep(10)
!echo "Top-K recommendations for user_id=1, segment=0 (from the trained retrieval model):"
!curl -s -X POST http://localhost:8501/v1/models/retrieval:predict -d '{"inputs": {"user_id": ["1"], "segment": [0]}}' | head -c 700 || cat /content/tfserve.log

## 4. TorchServe — serve the **trained** PyTorch ranker\nLoads `models/ranker.pt` (the exact file evaluated at +6.3%). Needs Java.

In [ ]:
!apt-get install -y -qq openjdk-17-jdk >/dev/null
!rm -rf model_store && mkdir -p model_store
!torch-model-archiver --model-name ranker --version 1.0 \
    --serialized-file models/ranker.pt --handler serving/torchserve_handler.py \
    --extra-files "src/ranking_torch.py,models/id_maps.json" --export-path model_store -f
import subprocess, time
subprocess.Popen(["torchserve","--start","--ncs","--model-store","model_store",
                  "--models","ranker=ranker.mar","--disable-token-auth"],
                 stdout=open("/content/ts.log","w"), stderr=subprocess.STDOUT)
time.sleep(28)
!echo "Ranker score for (user=0, movie=12, seg=3):"
!curl -s -X POST http://localhost:8080/predictions/ranker -H "Content-Type: application/json" -d '{"user":0,"movie":12,"seg":3}' || tail -20 /content/ts.log

## 5. Triton / ONNX — serve the **trained** ranker as ONNX
Triton's server needs Docker/GPU-VM (not in Colab), so we load the committed `models/ranker.onnx`
with ONNX Runtime to prove it serves, and show the Triton `config.pbtxt` you'd deploy.

In [ ]:
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession("triton/model_repository/ranker_onnx/1/model.onnx", providers=["CPUExecutionProvider"])
out = sess.run(None, {"user": np.array([0],np.int64), "movie": np.array([12],np.int64), "seg": np.array([3],np.int64)})
print("ONNX Runtime score (same trained ranker):", out[0].ravel())
print("\n--- Triton config.pbtxt you'd deploy on a Triton box ---")
!cat triton/model_repository/ranker_onnx/config.pbtxt

## 6. Continuity — one model, one eval
The three servers above all load the **same trained artifacts** from `models/` — nothing was
retrained. The offline A/B metric (**+6.3% NDCG@10**, ranker vs popularity, 609 users) was
measured **once** in the training half on this exact `ranker.pt`, so there is no second/duplicate
evaluation.

### Production notes (millions of users)
- **Registry, not git:** here `models/` stands in for a model registry. In production you'd push
  the trained artifacts to MLflow / Vertex Model Registry / S3 and the serving layer pulls a
  versioned model — the training and serving jobs never share a filesystem.
- **Train on a schedule, serve continuously:** the TFX pipeline (below) retrains + validates on
  fresh data on a cadence (e.g. nightly); serving keeps loading the last *pushed* model. You do
  not retrain per request or per deploy.
- **Retrieval + ranking at scale:** the two-tower retrieval narrows millions of items to a few
  hundred with an ANN index; the ranker re-scores only that shortlist — that's what keeps
  real-time latency bounded.

### About TFX / Kubeflow (why it's not in this notebook)
TFX 1.15 has **no Python 3.12 build**, and Colab is 3.12 — so the TFX training pipeline can't run
in this runtime. It needs an **x86 Linux + Python ≤3.11** environment. Options, cleanest first:
1. A separate Colab using **`condacolab`** to make a Python 3.10 env (free, x86) — ask and I'll
   build that notebook.
2. A cloud VM / GitHub Codespaces with Python 3.10.
3. Ship the **compiled Kubeflow v2 pipeline spec** as the deliverable and run it on Vertex AI.